<a href="https://colab.research.google.com/github/aymuos/endgame/blob/main/03_PCMCI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook runs the pcmci family of causal discovery algorithms on the correctedv3 dataset

In [4]:
from google.colab import drive
drive.mount('/content/drive')

KeyboardInterrupt: 

In [ ]:
DATASET_PATH = '/content/drive/MyDrive/ml/CORRECTEDv3/all_cities_combined_v3.parquet'

In [3]:
!pip install tigramite

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 314.7/314.7 kB 20.6 MB/s eta 0:00:00


# Objective

- PCMCI+ — single pooled temporal causal discovery ,  Output : Directed lagged graph
- J-PCMCI+ — joint multi-city temporal causal discovery , Output : Joint graph

## PCMCI+ for Individual Cities

In [2]:

import pandas as pd
import numpy as np
import tigramite
from tigramite import data_processing as pp
from tigramite.pcmci import PCMCI
from tigramite.independence_tests.parcorr import ParCorr

city_datasets = {
    'Shanghai': '/content/drive/MyDrive/ml/CORRECTEDv2/batch_scm_shanghai_v5.parquet',
    'Hangzhou': '/content/drive/MyDrive/ml/CORRECTEDv2/batch_scm_hangzhou_v5.parquet',
    'Chongqing': '/content/drive/MyDrive/ml/CORRECTEDv2/batch_scm_chongqing_v5.parquet'
}

# Define the features requested (same as before)
requested_features_individual = [
    'workload_causal', 'workload_capped', 'high_load', 'overloaded',
    'pickup_destination_distance', 'batch_size', 'batch_rank_dispatch',
    'batch_rank_capped', 'late_batch', 'extreme_batch', 'hour_sin', 'hour_cos',
    'day_sin', 'day_cos', 'is_weekend', 'is_holiday', 'is_holiday_eve',
    'spatial_congestion_index_daily', 'spatial_congestion_index_rolling7',
    'spatial_congestion_norm', 'courier_local_load', 'WSI', 'precipitation',
    'temperature_2m', 'windspeed_10m', 'is_trajectory_available', 'typecode_cb', 'eta_mins'
]

for city, path in city_datasets.items():
    print(f"\nProcessing {city} dataset: {path}")

    # 1. Load the dataset
    df_city = pd.read_parquet(path)

    # Handle wildcard for typecode_grouped_* specific to this dataframe
    grouped_features_city = [col for col in df_city.columns if col.startswith('typecode_grouped_')]
    current_requested_features = list(requested_features_individual)
    current_requested_features.extend(grouped_features_city)

    # Check which features actually exist in the dataframe to avoid KeyErrors
    features_city = [f for f in current_requested_features if f in df_city.columns]
    missing_city = set(current_requested_features) - set(features_city)
    if missing_city:
        print(f"Skipping missing columns for {city}: {missing_city}")

    # Filter dataframe and handle missing values for PCMCI
    selected_df_city = df_city[features_city].dropna().reset_index(drop=True)

    if selected_df_city.empty:
        print(f"Warning: DataFrame for {city} is empty after dropping NaNs. Skipping PCMCI+ for this city.")
        continue

    # 2. Initialize Tigramite dataframe object
    var_names_city = selected_df_city.columns.tolist()
    data_values_city = selected_df_city.values
    link_matrix_data_city = pp.DataFrame(data_values_city, var_names=var_names_city)

    # 3. Initialize and Run PCMCI+
    # Fixed: Changed significance='ait' to 'analytic'
    parcorr_city = ParCorr(significance='analytic')
    pcmci_city = PCMCI(dataframe=link_matrix_data_city, cond_ind_test=parcorr_city, verbosity=1)

    # Run pcmci_plus
    results_city = pcmci_city.run_pcmciplus(tau_max=2, pc_alpha=0.05)

    # Display summary results
    print(f"\nSignificant links for {city}:")
    pcmci_city.print_significant_links(
        p_matrix=results_city['p_matrix'],
        val_matrix=results_city['val_matrix'],
        alpha_level=0.05
    )

ModuleNotFoundError: No module named 'tigramite'